In [1]:
# CSV files are removed below before querying.

In [2]:
import re
import subprocess
from pathlib import Path

import pandas as pd

systems = ['tioga', #'frontier', 
           'tuo']
cluster_names = {'tioga': 'tioga', 'frontier': 'frontier', 'tuo': 'tuolumne'}
rocm_versions = ['rocm720', 'rocm642']
hip_versions = {'rocm720': '7.2.0', 'rocm642': '6.4.2'}

def workspace_paths(application, require_cali=False):
    paths = []
    for system in systems:
        for rocm_version in rocm_versions:
            path = Path('..') / 'wkp' / f'{rocm_version}-{system}' / application
            if not path.exists():
                continue
            if require_cali and not any(path.glob('**/*.cali')):
                continue
            paths.append(f'{path}/')
    return paths

for csv_file in Path('.').glob('*.csv'):
    csv_file.unlink()

def append_queries(queries, output_csv):
    frames = []
    for query in queries:
        option_start = next(
            (idx for idx, arg in enumerate(query[2:], start=2) if str(arg).startswith('--')),
            len(query),
        )
        if option_start == 2:
            continue
        result = subprocess.run(query, check=False, capture_output=True, text=True)
        if result.returncode != 0:
            empty_query = any(
                message in result.stderr
                for message in ['No Caliper files found', 'No regions matched']
            )
            if empty_query:
                print(f'Skipping empty query: {result.stderr.strip()}')
                continue
            result.check_returncode()
        csv_name = next(
            line.strip()
            for line in reversed(result.stdout.splitlines())
            if line.strip().endswith('.csv')
        )
        frames.append(pd.read_csv(csv_name))
        Path(csv_name).unlink()
    if frames:
        df = pd.concat(frames, ignore_index=True)
        header = not Path(output_csv).exists()
        df.to_csv(output_csv, mode='a', header=header, index=False)

lammps_timing_re = re.compile(
    r'^(Pair|Neigh|Comm|Output|Modifier|Kspace|Bond)\s+\|\s+'
    r'\S+\s+\|\s+(?P<avg_time>\S+)\s+\|'
)

def append_lammps_times(comp_csv, comm_csv):
    comp_rows = []
    comm_rows = []
    for system in systems:
        for rocm_version in rocm_versions:
            workspace = Path('..') / 'wkp' / f'{rocm_version}-{system}' / 'lammps'
            if not workspace.exists():
                continue
            for out_file in workspace.glob('workspace/experiments/lammps/*/*/*.out'):
                timings = {}
                for line in out_file.read_text(errors='ignore').splitlines():
                    match = lammps_timing_re.match(line)
                    if match:
                        timings[match.group(1)] = float(match.group('avg_time'))
                if not timings:
                    continue
                row = {
                    'cluster': cluster_names[system],
                    'application_name': 'lammps',
                    'packages.dependencies.hip.version': hip_versions[rocm_version],
                }
                communication = timings.get('Comm', 0)
                computation = sum(value for name, value in timings.items() if name != 'Comm')
                if computation:
                    comp_rows.append({**row, 'Avg time/rank (exc)': computation})
                if communication:
                    comm_rows.append({**row, 'Avg time/rank (exc)': communication})
    for rows, csv_file in [(comp_rows, comp_csv), (comm_rows, comm_csv)]:
        if rows:
            pd.DataFrame(rows).to_csv(
                csv_file,
                mode='a',
                header=not Path(csv_file).exists(),
                index=False,
            )


In [3]:
computation_queries = [
    [
        '../bin/benchpark', 'query',
        *workspace_paths('amg2023', require_cali=True),
        '--query-regions-byname', 'Problem',
        '--metric', 'Avg time/rank (exc)',
        '--metadata-columns', 'packages.dependencies.hip.version',
        '--exclude-regions', 'MPI_',
    ],
    [
        '../bin/benchpark', 'query',
        *workspace_paths('kripke', require_cali=True),
        '--query-regions-byname', 'Solve',
        '--metric', 'Avg time/rank (exc)',
        '--metadata-columns', 'packages.dependencies.hip.version',
        '--exclude-regions', 'MPI_',
    ],
    [
        '../bin/benchpark', 'query',
        *workspace_paths('laghos', require_cali=True),
        '--query-regions-byname', 'SolveVelocity-ForcePA', 'SolveEnergy-ForcePA', 'SolveVelocity-CGVMass', 'QUpdate-UpdateQuadratureData',
        '--metric', 'Avg time/rank (exc)',
        '--metadata-columns', 'packages.dependencies.hip.version',
        '--exclude-regions', 'MPI_', 'SolveEnergy-CGEMass',
    ],
]

append_queries(computation_queries, 'computation-time.csv')

In [4]:
communication_queries = [
    [
        '../bin/benchpark', 'query',
        *workspace_paths('amg2023', require_cali=True),
        '--query-regions-byname', 'Problem',
        '--filter-regions-byname', 'MPI_',
        '--metric', 'Avg time/rank (exc)',
        '--metadata-columns', 'packages.dependencies.hip.version',
    ],
    [
        '../bin/benchpark', 'query',
        *workspace_paths('kripke', require_cali=True),
        '--query-regions-byname', 'Solve',
        '--filter-regions-byname', 'MPI_',
        '--metric', 'Avg time/rank (exc)',
        '--metadata-columns', 'packages.dependencies.hip.version',
    ],
    [
        '../bin/benchpark', 'query',
        *workspace_paths('laghos', require_cali=True),
        '--query-regions-byname', 'SolveVelocity-ForcePA', 'SolveEnergy-ForcePA', 'SolveVelocity-CGVMass', 'QUpdate-UpdateQuadratureData',
        '--filter-regions-byname', 'MPI_',
        '--metric', 'Avg time/rank (exc)',
        '--metadata-columns', 'packages.dependencies.hip.version',
    ],
]

append_queries(communication_queries, 'communication-time.csv')
append_lammps_times('computation-time.csv', 'communication-time.csv')

In [5]:
! ls *.csv

communication-time.csv computation-time.csv
